# Crystalline: Qwen2.5 Optimization Pipeline

This notebook runs the **Crystalline** AI framework on Qwen2.5 models.

**Pipeline:**
1. Mount Google Drive for model caching & checkpointing
2. Download Qwen2.5 from HuggingFace
3. Baseline benchmark (FP16)
4. Distill into a Crystalline student
5. Crystallize weights to {-1, 0, 1}
6. Final benchmark & telemetry

**Recommended Runtime:** GPU (T4 or better)

## Section 1: Mount Google Drive

In [ ]:
# ============================================================
# Section 1: Mount Google Drive for Model Caching
# ============================================================
from google.colab import drive
import os

DRIVE_PATH = "/content/drive/MyDrive/CrystallineCache"
os.makedirs(DRIVE_PATH, exist_ok=True)
drive.mount("/content/drive", force_remount=True)

print(f"Drive mounted. Cache directory: {DRIVE_PATH}")

## Section 2: Install Dependencies

In [ ]:
# ============================================================
# Section 2: Install Dependencies
# ============================================================
!pip install -q transformers accelerate bitsandbytes
!pip install -q torch psutil tqdm huggingface_hub datasets matplotlib pandas

print("Dependencies installed.")

## Section 3: Clone / Import Crystalline Framework

In [ ]:
# ============================================================
# Section 3: Import Crystalline and Utilities
# ============================================================
import sys, os, json, time, math
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Add repo path so we can import crystalline
# If running standalone, clone the repo first:
# !git clone https://github.com/YOUR_REPO.git /content/crystalline_repo
# sys.path.insert(0, '/content/crystalline_repo')

try:
    from crystalline import (
        CrystallineModel, CrystallineConfig,
        crystallize_module, crystallization_penalty,
        tropical_matmul, tropical_dot_product,
        sheffer_nand, tropical_to_sheffer,
    )
    from crystalline.train import (
        train_crystalline_model, TextDataset, generate_synthetic_data,
        crystalline_distillation_loss,
    )
    from qwen_optimizer.benchmark import BenchmarkSuite, BenchmarkResult
    from qwen_optimizer.telemetry import TelemetryLogger, TelemetryEntry
    from qwen_optimizer.download import ModelCache, check_disk_space
    print("Crystalline framework imported successfully.")
except ImportError as e:
    print(f"Import error: {e}")
    print("Please ensure the crystalline package is available in the Python path.")

## Section 4: Configuration

In [ ]:
# ============================================================
# Section 4: Configuration
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # Start small; change to 1.5B or 3B if you have VRAM
CACHE_DIR = os.path.join(DRIVE_PATH, "models")
OUTPUT_DIR = os.path.join(DRIVE_PATH, "output")
TELEMETRY_FILE = os.path.join(DRIVE_PATH, "telemetry.json")

# Crystalline student config
STUDENT_CONFIG = CrystallineConfig(
    vocab_size=151936,   # Qwen2.5 vocab size
    d_model=256,
    num_layers=4,
    num_heads=4,
    d_ff=512,
    max_seq_len=512,
    dropout=0.1,
    use_delta_net=False,  # Standard attention for Qwen2.5 baseline
    num_experts=1,        # No MoE for small student
    top_k=1,
)

# Training hyperparameters
EPOCHS = 1
BATCH_SIZE = 2
LR = 5e-5
TEMPERATURE = 2.0
ALPHA = 0.5
CRYSTALLIZATION_WEIGHT = 0.005
NUM_SYNTHETIC_SAMPLES = 50
MAX_LENGTH = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Student params: d_model={STUDENT_CONFIG.d_model}, layers={STUDENT_CONFIG.num_layers}")

## Section 5: Download & Cache Model

In [ ]:
# ============================================================
# Section 5: Model Download with Drive Caching
# ============================================================
cache = ModelCache(CACHE_DIR)
model_path = cache.get_model_path(MODEL_NAME)
print(f"Model cached at: {model_path}")

## Section 6: Baseline Benchmark (FP16)

In [ ]:
# ============================================================
# Section 6: Baseline Benchmark
# ============================================================
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("Loading baseline model in FP16...")
t0 = time.time()
teacher = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else "cpu",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")

bench = BenchmarkSuite(tokenizer, device=DEVICE)
logger = TelemetryLogger(TELEMETRY_FILE)

baseline_result = bench.run_inference_benchmark(
    teacher, "baseline_fp16", "fp16", load_time=load_time
)
logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=baseline_result.stage,
    model_name=MODEL_NAME,
    quantization=baseline_result.quantization,
    vram_mb=baseline_result.vram_mb,
    tokens_per_sec_prefill=baseline_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=baseline_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=baseline_result.latency_ttft_ms,
    latency_tpot_ms=baseline_result.latency_tpot_ms,
    notes=f"Load time: {load_time:.1f}s",
))

print(f"\n--- Baseline Results ---")
print(f"VRAM: {baseline_result.vram_mb:.1f} MB")
print(f"Prefill: {baseline_result.tokens_per_sec_prefill:.1f} tok/s")
print(f"Decode: {baseline_result.tokens_per_sec_decode:.1f} tok/s")
print(f"TTFT: {baseline_result.latency_ttft_ms:.1f} ms")
print(f"TPOT: {baseline_result.latency_tpot_ms:.1f} ms")

## Section 7: Create Crystalline Student

In [ ]:
# ============================================================
# Section 7: Create Crystalline Student
# ============================================================
# Match vocab size to teacher
STUDENT_CONFIG.vocab_size = teacher.config.vocab_size

student = CrystallineModel(STUDENT_CONFIG)
student = student.to(DEVICE)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())
print(f"Teacher params: {teacher_params/1e6:.1f}M")
print(f"Student params: {student_params/1e6:.1f}M")
print(f"Compression ratio: {teacher_params / student_params:.1f}x")

# Sanity check forward pass
with torch.no_grad():
    test_ids = torch.randint(0, tokenizer.vocab_size, (1, 16), device=DEVICE)
    logits = student(test_ids)
    print(f"Student forward pass OK: {logits.shape}")

## Section 8: Generate Synthetic Dataset

In [ ]:
# ============================================================
# Section 8: Synthetic Dataset from Teacher
# ============================================================
print("Generating synthetic dataset from teacher...")
synthetic_texts = generate_synthetic_data(
    teacher=teacher,
    tokenizer=tokenizer,
    num_samples=NUM_SYNTHETIC_SAMPLES,
    max_length=MAX_LENGTH,
    device=DEVICE,
)
train_dataset = TextDataset(synthetic_texts, tokenizer, max_length=MAX_LENGTH)
print(f"Dataset size: {len(train_dataset)} samples")
for i, text in enumerate(synthetic_texts[:3]):
    print(f"  Sample {i+1}: {text[:80]}...")

## Section 9: Distill Teacher into Crystalline Student

In [ ]:
# ============================================================
# Section 9: Distillation Training
# ============================================================
print("\nStarting distillation training...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

trained_student = train_crystalline_model(
    teacher=teacher,
    student=student,
    tokenizer=tokenizer,
    dataset=train_dataset,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    device=DEVICE,
    temperature=TEMPERATURE,
    alpha=ALPHA,
    crystallization_weight=CRYSTALLIZATION_WEIGHT,
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
)

# Benchmark distilled student
dist_result = bench.run_inference_benchmark(
    trained_student, "distilled", "crystalline_fp16",
    notes=f"epochs={EPOCHS}, alpha={ALPHA}, T={TEMPERATURE}",
)
logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=dist_result.stage,
    model_name="CrystallineStudent",
    quantization=dist_result.quantization,
    vram_mb=dist_result.vram_mb,
    tokens_per_sec_prefill=dist_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=dist_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=dist_result.latency_ttft_ms,
    latency_tpot_ms=dist_result.latency_tpot_ms,
    notes=dist_result.notes,
))

print(f"\n--- Distilled Results ---")
print(f"VRAM: {dist_result.vram_mb:.1f} MB")
print(f"Decode: {dist_result.tokens_per_sec_decode:.1f} tok/s")
print(f"TPOT: {dist_result.latency_tpot_ms:.1f} ms")

## Section 10: Crystallize Weights

In [ ]:
# ============================================================
# Section 10: Crystallization
# ============================================================
pre_penalty = sum(crystallization_penalty(p) for p in trained_student.parameters())
print(f"Crystallization penalty BEFORE: {pre_penalty.item():.4f}")

trained_student.crystallize()

post_penalty = sum(crystallization_penalty(p) for p in trained_student.parameters())
print(f"Crystallization penalty AFTER: {post_penalty.item():.4f}")

# Count discrete values
total_params = sum(p.numel() for p in trained_student.parameters())
neg1 = sum((p == -1).sum().item() for p in trained_student.parameters())
zeros = sum((p == 0).sum().item() for p in trained_student.parameters())
pos1 = sum((p == 1).sum().item() for p in trained_student.parameters())
print(f"Weight distribution: -1: {neg1/total_params*100:.2f}%, 0: {zeros/total_params*100:.2f}%, +1: {pos1/total_params*100:.2f}%")

# Benchmark crystallized model
cryst_result = bench.run_inference_benchmark(
    trained_student, "crystallized", "ternary",
    notes="weights in {-1,0,1}",
)
logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=cryst_result.stage,
    model_name="CrystallizedStudent",
    quantization=cryst_result.quantization,
    vram_mb=cryst_result.vram_mb,
    tokens_per_sec_prefill=cryst_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=cryst_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=cryst_result.latency_ttft_ms,
    latency_tpot_ms=cryst_result.latency_tpot_ms,
    notes=cryst_result.notes,
))

print(f"\n--- Crystallized Results ---")
print(f"VRAM: {cryst_result.vram_mb:.1f} MB")
print(f"Decode: {cryst_result.tokens_per_sec_decode:.1f} tok/s")
print(f"TPOT: {cryst_result.latency_tpot_ms:.1f} ms")

## Section 11: Telemetry Summary & Visualization

In [ ]:
# ============================================================
# Section 11: Summary & Plots
# ============================================================
logger.summary()

chart_path = os.path.join(DRIVE_PATH, "benchmark_chart.png")
logger.plot_comparison(save_path=chart_path)

csv_path = os.path.join(DRIVE_PATH, "telemetry.csv")
logger.export_csv(csv_path)

# Save model
model_path = os.path.join(DRIVE_PATH, "crystallized_student.pt")
torch.save(trained_student.state_dict(), model_path)
print(f"\nModel saved to: {model_path}")
print(f"Telemetry JSON: {TELEMETRY_FILE}")
print(f"Telemetry CSV: {csv_path}")
print(f"Chart: {chart_path}")
print("\nDone!")

## Section 12: Sheffer NAND Logic Demo

In [ ]:
# ============================================================
# Section 12: Sheffer NAND Demo
# ============================================================
a = torch.tensor([0.0, 0.0, 1.0, 1.0], device=DEVICE)
b = torch.tensor([0.0, 1.0, 0.0, 1.0], device=DEVICE)
nand_out = sheffer_nand(a, b)
print("Sheffer NAND (functionally complete):")
print(f"a     = {a.tolist()}")
print(f"b     = {b.tolist()}")
print(f"NAND  = {nand_out.tolist()}")
print("(Any boolean circuit can be built from NAND gates)")